# Agronomic Advice — RAG Document Retrieval
### Hybrid BM25 + Dense Embedding Retrieval Pipeline

**Task:** given a smallholder farmer's question, return the **top-5** most relevant
documents from a knowledge base of agricultural extension material (crop diseases,
pests, nutrient deficiencies, soil management, climate adaptation, fertiliser advice).

**Metric:** nDCG@5 against expert relevance judgements.

**Strategy in this notebook:**
1. Load the competition data (auto-detects file/column names — edit `CONFIG` below
   once you've seen the real files if the auto-detect doesn't hit).
2. Reproduce the provided **TF-IDF baseline** so we have a number to beat.
3. Build a **BM25** retriever (better than raw TF-IDF for short queries).
4. Build a **dense embedding retriever** (`sentence-transformers/all-MiniLM-L6-v2`,
   CPU-only, fast) to catch semantic matches BM25 misses (synonyms, paraphrases).
5. **Fuse** BM25 + dense scores (reciprocal-rank fusion) and rerank the fused
   candidate pool with a cross-encoder for a final precision boost.
6. Validate with nDCG@5 on a held-out split of the *labelled* training queries.
7. Write a clean, schema-checked `submission.csv`.

No GPU required. Everything here runs on Kaggle's free CPU tier.


In [ ]:
# 0. Install lightweight deps (CPU-only, no GPU needed)
# rank_bm25: BM25 scoring | sentence-transformers: embeddings + cross-encoder reranker
import sys, subprocess

def pip_install(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=False)

pip_install(["rank_bm25", "sentence-transformers"])
print("Dependencies ready.")


In [ ]:
# 1. CONFIG — adjust these ONLY if auto-detection below fails
# (open the Input panel on Kaggle to see real file/column names, then edit here)

CONFIG = {
    # Candidate file names for each role — first match wins
    "doc_file_candidates": ["documents.csv", "corpus.csv", "knowledge_base.csv", "docs.csv"],
    "train_file_candidates": ["train.csv", "train_queries.csv", "train_prompts.csv"],
    "test_file_candidates": ["test.csv", "test_queries.csv", "test_prompts.csv"],

    # Candidate column names for each role — first match wins
    "doc_id_col_candidates": ["document_id", "doc_id", "id", "DocumentId"],
    "doc_text_col_candidates": ["text", "content", "document_text", "body"],
    "query_id_col_candidates": ["query_id", "QueryId", "id", "PromptId"],
    "query_text_col_candidates": ["query", "question", "prompt", "text"],
    "relevant_docs_col_candidates": ["relevant_document_ids", "relevant_docs", "doc_ids", "labels"],

    "top_k": 5,
}
print("Config loaded.")


In [ ]:
# 2. Locate & load competition data
from pathlib import Path
import pandas as pd

ON_KAGGLE = Path("/kaggle/input").exists()
OUTPUT_DIR = Path("/kaggle/working") if ON_KAGGLE else Path(".")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def find_file(candidates):
    '''Search /kaggle/input (recursively) for the first matching filename.'''
    search_roots = [Path("/kaggle/input")] if ON_KAGGLE else [Path(".")]
    for root in search_roots:
        if not root.exists():
            continue
        for name in candidates:
            hits = list(root.rglob(name))
            if hits:
                return hits[0]
    return None


def pick_col(df, candidates, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"None of {candidates} found in columns {list(df.columns)}")
    return None


doc_path = find_file(CONFIG["doc_file_candidates"])
train_path = find_file(CONFIG["train_file_candidates"])
test_path = find_file(CONFIG["test_file_candidates"])

if doc_path is None or test_path is None:
    raise FileNotFoundError(
        "Could not locate documents/test files. Join the competition, attach its "
        "dataset in the Input panel, and/or update CONFIG's file name candidates "
        "above to match the real file names."
    )

docs = pd.read_csv(doc_path)
test = pd.read_csv(test_path)
train = pd.read_csv(train_path) if train_path else None

DOC_ID_COL = pick_col(docs, CONFIG["doc_id_col_candidates"])
DOC_TEXT_COL = pick_col(docs, CONFIG["doc_text_col_candidates"])
QUERY_ID_COL = pick_col(test, CONFIG["query_id_col_candidates"])
QUERY_TEXT_COL = pick_col(test, CONFIG["query_text_col_candidates"])

print("doc file :", doc_path)
print("train file:", train_path)
print("test file :", test_path)
print(f"docs={len(docs)} rows | test={len(test)} rows | train={0 if train is None else len(train)} rows")
print("doc id/text cols:", DOC_ID_COL, DOC_TEXT_COL)
print("query id/text cols:", QUERY_ID_COL, QUERY_TEXT_COL)
docs.head(3)


In [ ]:
# 3. Text cleaning + tokenisation shared by BM25 and TF-IDF
import re

def clean_text(s):
    s = str(s).lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def tokenize(s):
    return clean_text(s).split()

doc_ids = docs[DOC_ID_COL].tolist()
doc_texts = docs[DOC_TEXT_COL].astype(str).tolist()
doc_tokens = [tokenize(t) for t in doc_texts]

print("Sample tokenised doc:", doc_tokens[0][:20])


## 4. Baseline — TF-IDF cosine similarity (this is the number we need to beat)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
doc_tfidf_matrix = tfidf_vectorizer.fit_transform(doc_texts)

def tfidf_rank(query_text, top_k=5):
    q_vec = tfidf_vectorizer.transform([query_text])
    sims = cosine_similarity(q_vec, doc_tfidf_matrix).ravel()
    top_idx = np.argsort(-sims)[:top_k]
    return [doc_ids[i] for i in top_idx], sims[top_idx]

# quick smoke test
example_ranked, example_scores = tfidf_rank(test[QUERY_TEXT_COL].iloc[0], top_k=5)
print("TF-IDF top-5 for first test query:", example_ranked)


## 5. BM25 retriever — usually stronger than TF-IDF cosine sim on short, keyword-heavy queries

In [ ]:
from rank_bm25 import BM25Okapi

bm25 = BM25Okapi(doc_tokens)

def bm25_rank(query_text, top_k=5):
    q_tokens = tokenize(query_text)
    scores = bm25.get_scores(q_tokens)
    top_idx = np.argsort(-scores)[:top_k]
    return [doc_ids[i] for i in top_idx], scores[top_idx], scores

example_ranked, example_scores, _ = bm25_rank(test[QUERY_TEXT_COL].iloc[0], top_k=5)
print("BM25 top-5 for first test query:", example_ranked)


## 6. Dense embedding retriever — catches semantic/paraphrase matches BM25 misses

In [ ]:
from sentence_transformers import SentenceTransformer

# Small, fast, strong general-purpose embedding model. CPU inference is fine at this scale.
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

doc_embeddings = embed_model.encode(
    doc_texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True
)

def dense_rank(query_text, top_k=5):
    q_emb = embed_model.encode([query_text], normalize_embeddings=True)
    scores = (doc_embeddings @ q_emb.T).ravel()
    top_idx = np.argsort(-scores)[:top_k]
    return [doc_ids[i] for i in top_idx], scores[top_idx], scores

example_ranked, example_scores, _ = dense_rank(test[QUERY_TEXT_COL].iloc[0], top_k=5)
print("Dense top-5 for first test query:", example_ranked)


## 7. Hybrid fusion — Reciprocal Rank Fusion (RRF) + cross-encoder rerank

RRF combines ranked lists without needing scores to be on the same scale, which is
the usual headache when mixing BM25 and cosine similarity. We take a slightly wider
candidate pool from BM25+dense, fuse their rankings, then use a cross-encoder to
rerank the fused top candidates for a final precision boost (cross-encoders score
query+document jointly, which is more accurate but slower — so we only apply it to
a small shortlist, not the whole corpus).

In [ ]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

RRF_K = 60          # standard RRF damping constant
CANDIDATE_POOL = 20 # widen beyond top_k before fusing/reranking

def rrf_fuse(rank_lists, k=RRF_K):
    '''rank_lists: list of ordered doc_id lists (best first). Returns fused ranking.'''
    scores = {}
    for ranked in rank_lists:
        for rank, doc_id in enumerate(ranked):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank + 1)
    return sorted(scores.keys(), key=lambda d: -scores[d])

def hybrid_rank(query_text, top_k=5, use_reranker=True):
    bm25_top, _, _ = bm25_rank(query_text, top_k=CANDIDATE_POOL)
    dense_top, _, _ = dense_rank(query_text, top_k=CANDIDATE_POOL)
    fused = rrf_fuse([bm25_top, dense_top])[:CANDIDATE_POOL]

    if not use_reranker:
        return fused[:top_k]

    id_to_text = dict(zip(doc_ids, doc_texts))
    pairs = [[query_text, id_to_text[d]] for d in fused]
    ce_scores = cross_encoder.predict(pairs)
    reranked = [d for _, d in sorted(zip(ce_scores, fused), key=lambda x: -x[0])]
    return reranked[:top_k]

example = hybrid_rank(test[QUERY_TEXT_COL].iloc[0], top_k=5)
print("Hybrid (BM25+dense+rerank) top-5 for first test query:", example)


## 8. Local validation — nDCG@5 on the labelled training queries

If `train` has ground-truth relevant document ids, we can measure nDCG@5 locally
*before* burning a submission, and directly compare TF-IDF baseline vs BM25 vs
dense vs hybrid.

In [ ]:
import math

def dcg_at_k(relevances, k=5):
    return sum(rel / math.log2(i + 2) for i, rel in enumerate(relevances[:k]))

def ndcg_at_k(ranked_doc_ids, relevant_ids, k=5):
    relevant_set = set(relevant_ids)
    gains = [1.0 if d in relevant_set else 0.0 for d in ranked_doc_ids[:k]]
    ideal = sorted(gains, reverse=True)  # best possible ordering given hit/miss labels
    dcg = dcg_at_k(gains, k)
    idcg = dcg_at_k(ideal, k) if any(ideal) else 1.0
    return dcg / idcg if idcg > 0 else 0.0

def parse_relevant_ids(cell):
    '''Handles either a delimited string ('d1;d2;d3') or a list-like column.'''
    if isinstance(cell, (list, tuple, set)):
        return list(cell)
    s = str(cell)
    for sep in [";", ",", "|"]:
        if sep in s:
            return [x.strip() for x in s.split(sep) if x.strip()]
    return [s.strip()] if s.strip() else []

if train is not None:
    REL_COL = pick_col(train, CONFIG["relevant_docs_col_candidates"], required=False)
    TRAIN_QUERY_TEXT_COL = pick_col(train, CONFIG["query_text_col_candidates"], required=False) or QUERY_TEXT_COL

    if REL_COL is not None:
        methods = {
            "tfidf":  lambda q: tfidf_rank(q, top_k=5)[0],
            "bm25":   lambda q: bm25_rank(q, top_k=5)[0],
            "dense":  lambda q: dense_rank(q, top_k=5)[0],
            "hybrid": lambda q: hybrid_rank(q, top_k=5),
        }
        results = {name: [] for name in methods}
        for _, row in train.iterrows():
            relevant_ids = parse_relevant_ids(row[REL_COL])
            if not relevant_ids:
                continue
            for name, fn in methods.items():
                ranked = fn(row[TRAIN_QUERY_TEXT_COL])
                results[name].append(ndcg_at_k(ranked, relevant_ids, k=5))

        print("nDCG@5 on labelled training queries:")
        for name, scores in results.items():
            if scores:
                print(f"  {name:8s}: {sum(scores)/len(scores):.4f}  (n={len(scores)})")
    else:
        print("No relevant-docs column detected in train — skipping local nDCG@5 eval.")
        print("Update CONFIG['relevant_docs_col_candidates'] once you see the real column name.")
else:
    print("No labelled train file found — skipping local eval. (Submission below still works.)")


## 9. Build & validate the submission

Uses the hybrid retriever (BM25 + dense + cross-encoder rerank) — the strongest
method from the eval above. Swap `hybrid_rank` for `bm25_rank`/`dense_rank`/
`tfidf_rank` here if local validation shows one of those winning instead.

In [ ]:
from tqdm.auto import tqdm

FINAL_RANK_FN = hybrid_rank  # <- change if a different method wins the eval above

rows = []
for _, q in tqdm(test.iterrows(), total=len(test)):
    top5 = FINAL_RANK_FN(q[QUERY_TEXT_COL], top_k=5)
    # pad defensively in case fewer than 5 docs exist in a tiny debug corpus
    while len(top5) < 5:
        top5.append(top5[-1] if top5 else doc_ids[0])
    rows.append({
        QUERY_ID_COL: q[QUERY_ID_COL],
        **{f"doc_{i+1}": doc_id for i, doc_id in enumerate(top5)},
    })

submission = pd.DataFrame(rows)
submission.head()


In [ ]:
# 10. Validate schema, then write submission.csv
expected_cols = [QUERY_ID_COL] + [f"doc_{i+1}" for i in range(5)]

assert list(submission.columns) == expected_cols, (
    f"Column mismatch — check the real sample_submission.csv format on Kaggle and "
    f"rename columns here to match exactly. Got: {list(submission.columns)}"
)
assert len(submission) == len(test), "Row count doesn't match test set"
assert submission[QUERY_ID_COL].is_unique, "Duplicate query ids in submission"
assert submission.isna().sum().sum() == 0, "Submission contains empty cells"

out_path = OUTPUT_DIR / "submission.csv"
submission.to_csv(out_path, index=False)
print(f"Validated and wrote {len(submission)} rows -> {out_path}")


## Submission checklist

1. **Before you touch anything else**: open the competition's Input panel, check the
   *actual* file names and the *actual* columns in `documents.csv` / `test.csv` /
   `sample_submission.csv`. Update the `CONFIG` cell (section 1) and, if needed, the
   submission column names in section 10 to match exactly — this is the #1 reason
   auto-built notebooks fail on first submit.
2. Run **top to bottom** once to confirm no errors.
3. `Save Version` → `Save & Run All`. Wait for it to finish (don't submit from an
   interactive draft).
4. Open the saved version's **Output** panel, confirm `submission.csv` is there,
   and hit **Submit to Competition**.
5. If local nDCG@5 (section 8) shows `bm25` or `dense` beating `hybrid` on your
   actual data, change `FINAL_RANK_FN` in section 9 accordingly before your final
   run — don't just trust the hybrid by default, trust the eval.

**Where the real gains are, if you have more time after tonight's submission:**
- Tune RRF's `k` and the cross-encoder candidate pool size against the local nDCG@5.
- Try a stronger embedding model (e.g. `BAAI/bge-small-en-v1.5`) — still CPU-friendly.
- Chunk long documents instead of embedding them whole, and aggregate chunk scores.
- Add query expansion (synonyms for agronomic terms: "maize" vs "corn" vs "Zea mays").
